In [1]:
# show → code input visible by default
# hide-output → output hidden by default
# show hide-output → both (can combine on one line)

import pandas as pd
#from google.cloud import bigquery
from common_lib.sql import BigQueryConnector
from common_lib.export import export_notebook_html
import datetime as dt
import plotly.express as px


In [ ]:
refresh_data = False

In [ ]:
query_location = './sql/int_user_gems_flows.sql'

# Pulled in monthly chunks to avoid BQ's "Response too large to return" error on the
# full-year select * (likely a Storage API fallback to the size-capped tabledata.list path).
month_starts = pd.date_range('2021-01-01', '2025-12-01', freq='MS')
month_ranges = [(d.strftime('%Y-%m-%d'), (d + pd.offsets.MonthEnd(0)).strftime('%Y-%m-%d')) for d in month_starts]

bqc = BigQueryConnector()
for start_date, end_date in month_ranges:
    bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters={'start_date': start_date, 'end_date': end_date})

This query will process 0.0 KB when run.
Estimated query cost: $0.00
This query will process 0.0 KB when run.
Estimated query cost: $0.00
This query will process 0.0 KB when run.
Estimated query cost: $0.00
This query will process 0.0 KB when run.
Estimated query cost: $0.00
This query will process 0.0 KB when run.
Estimated query cost: $0.00
This query will process 13.1 MB when run.
Estimated query cost: $0.00
This query will process 182.2 MB when run.
Estimated query cost: $0.00
This query will process 441.3 MB when run.
Estimated query cost: $0.00
This query will process 1003.2 MB when run.
Estimated query cost: $0.01
This query will process 3.08 GB when run.
Estimated query cost: $0.02
This query will process 3.41 GB when run.
Estimated query cost: $0.02
This query will process 3.37 GB when run.
Estimated query cost: $0.02
This query will process 4.16 GB when run.
Estimated query cost: $0.03
This query will process 3.88 GB when run.
Estimated query cost: $0.03
This query will proce

In [4]:
# BQ NUMERIC columns land as slow, memory-heavy Decimal objects; cast to Int64
# (values are whole gem counts, confirmed via BQ schema + sampling) so the CSV
# write below can use pandas' vectorized numeric path instead of per-cell Python.
gem_count_columns = ['gems_earned', 'free_gems_inflow', 'paid_gems_inflow', 'gems_outflow']

month_csv_zip_paths = [f'./data/int_user_gems_flows_data_{start_date[:7]}.csv.zip' for start_date, _ in month_ranges]

if refresh_data:
    for (start_date, end_date), csv_zip_path in zip(month_ranges, month_csv_zip_paths):
        df_month = bqc.get(query='./sql/int_user_gems_flows.sql', is_path=True,
                            query_parameters={'start_date': start_date, 'end_date': end_date})
        df_month[gem_count_columns] = df_month[gem_count_columns].astype('Int64')
        df_month.to_csv(csv_zip_path, index=False, compression={'method': 'zip', 'compresslevel': 1})
        del df_month


In [5]:
query_location = './sql/km_trk_user_iap_revenue_monthly.sql'
parameters = {
}

bqc = BigQueryConnector()
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 210.7 MB when run.
Estimated query cost: $0.00


In [6]:
km_trk_user_iap_revenue_monthly = pd.DataFrame()

if refresh_data:
    km_trk_user_iap_revenue_monthly = bqc.get(query='./sql/km_trk_user_iap_revenue_monthly.sql', is_path=True, query_parameters=parameters)
    km_trk_user_iap_revenue_monthly.to_pickle('./data/km_trk_user_iap_revenue_monthly_data.pkl')
else:
    km_trk_user_iap_revenue_monthly = pd.read_pickle('./data/km_trk_user_iap_revenue_monthly_data.pkl')



In [7]:
km_trk_user_iap_revenue_monthly.to_csv('./data/km_trk_user_iap_revenue_monthly_data.csv', index=False)

In [8]:
km_trk_user_iap_revenue_monthly

,user_id,month_start_dt,max_active_dt,min_purchase_dt,bool_is_payer,total_iap_revenue_usd,total_iap_gems_only_revenue_usd,total_iap_non_gem_revenue_usd
0,FBAFD707E8910B7C,2025-07-01,2025-07-01,NaT,<NA>,NaN,NaN,NaN
1,FAFDC01FD9221B96,2025-07-01,2025-07-01,NaT,<NA>,NaN,NaN,NaN
2,FD38290FD318EDD3,2025-07-01,2025-07-01,NaT,<NA>,NaN,NaN,NaN
3,FF5D9D92B2F34AEA,2025-07-01,2025-07-01,NaT,<NA>,NaN,NaN,NaN
4,FAF3877EE1299E3C,2025-07-01,2025-07-01,NaT,<NA>,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
4921813,7B6AED91468137B9,2025-02-01,2025-02-28,2025-02-01,1,910.966451,6.229417,904.737034
4921814,6604E8BE89A8BD1B,2025-02-01,2025-02-28,2025-02-01,1,916.450000,284.840000,631.610000
4921815,A59223FB945462FC,2025-02-01,2025-02-28,2025-02-01,1,1033.260000,154.900000,878.360000
4921816,E1C1224E0B0F6624,2025-02-01,2025-02-28,2025-02-03,1,1064.670000,899.910000,164.760000


In [9]:
query_location = './sql/km_users_to_exclude.sql'
parameters = {
}

bqc = BigQueryConnector()
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 5.0 MB when run.
Estimated query cost: $0.00


In [10]:
km_users_to_exclude = pd.DataFrame()

if refresh_data:
    km_users_to_exclude = bqc.get(query='./sql/km_users_to_exclude.sql', is_path=True, query_parameters=parameters)
    km_users_to_exclude.to_pickle('./data/km_users_to_exclude_data.pkl')
else:
    km_users_to_exclude = pd.read_pickle('./data/km_users_to_exclude_data.pkl')



In [11]:
km_users_to_exclude.to_csv('./data/km_users_to_exclude_data.csv', index=False)

In [12]:
km_users_to_exclude

,userid,bool_user_to_exclude,min_exclude_date,max_exclude_date,had_energy_recharge_bug,hacked_free_gems_earned_energy,hacked_free_gems_earned_callacustomer,hacked_free_gems_earned_heartshop_storage,qa_tester_gifted_gems,spent_more_10K_free_gems_one_day,earned_more_1K_free_gems_one_day,gems_balance_200K_plus,is_tester
0,3B489B529C9D1AFB,1,2023-08-28,2023-12-14,0,0,1,1,0,1,1,0,0
1,19A9200A627636D2,1,2023-08-28,2023-08-28,0,0,0,1,0,1,1,0,0
2,D2233F6BC23880EA,1,2023-08-28,2023-08-28,0,0,1,1,0,0,1,0,0
3,35509EAB2C4091B8,1,2023-08-28,2023-09-28,0,0,0,0,0,1,0,0,0
4,DA920D61B7A6367F,1,2023-08-28,2023-08-29,0,0,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
45966,A70CF05F978E1C4E,1,2023-05-26,2023-05-26,0,0,0,0,0,1,0,1,0
45967,1134E1FE9EF4EE6,1,2023-05-26,2023-05-26,0,0,0,0,0,1,0,1,0
45968,C31EF7DFDDDF9AA6,1,2023-05-26,2023-05-26,0,0,0,0,0,1,0,1,0
45969,B9D0947D2318C5FD,1,2023-05-26,2023-05-27,0,0,0,0,0,1,0,1,0
